In [7]:
import requests
from bs4 import BeautifulSoup

url = "https://raw.githubusercontent.com/SimplifyJobs/Summer2026-Internships/dev/README.md"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'lxml') 
print("Data Fetched!")

Data Fetched!


In [31]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. Fetch the raw content
url = "https://raw.githubusercontent.com/SimplifyJobs/Summer2026-Internships/dev/README.md"
response = requests.get(url)
full_text = response.text

# 2. FIND THE START AND END OF THE DATA SCIENCE SECTION
# We use simple string finding because it's more reliable than BeautifulSoup for large files
start_keyword = "## 🤖 Data Science, AI & Machine Learning Internship Roles"
next_section_keyword = "## " # The next header (e.g., Quant or Hardware)

start_idx = full_text.find(start_keyword)
'''
we are findinfg the index of the section we are searching for and we are using
it as a starting index to travers untill the end index or
otherwise the next section/ end of file

'''
if start_idx != -1:
    # Find where the NEXT section starts to crop the text
    #str.find(sub[, start[, end]])...(##, index )
    end_idx = full_text.find(next_section_keyword, start_idx + len(start_keyword))
    # If there is no next section, go to the end of the file
    if end_idx == -1:
        end_idx = len(full_text)
        
    # slice the text to ONLY the Data Science section
    ds_content = full_text[start_idx:end_idx]
    
    # 3. PARSE ALL TABLES IN THIS CROP
    soup = BeautifulSoup(ds_content, 'html.parser')
    tables = soup.find_all('table')
    
    all_rows = []
    for table in tables:
        rows = table.find_all('tr')[1:]
        for row in rows:
            cols = row.find_all('td')
            cols[0].get_text(separator=" ").strip()
            
            keywords = ["scientist",
                        "science",
                        "market",
                        "marketing"
                        "analyst",
                        "business",
                        "intelligence"]
            keyStates = {
            "AL", "AK", "AZ", "AR", "CA",
            "CO", "CT", "DE", "FL", "GA",
            "HI", "ID", "IL", "IN", "IA",
            "KS", "KY", "LA", "ME", "MD",
            "MA", "MI", "MN", "MS", "MO",
            "MT", "NE", "NV", "NH", "NJ",
            "NM", "NY", "NC", "ND", "OH",
            "OK", "OR", "PA", "RI", "SC",
            "SD", "TN", "TX", "UT", "VT",
            "VA", "WA", "WV", "WI", "WY"
            }


            role = cols[1].get_text(separator=" ").strip().lower()
            
            location = cols[2].get_text(separator=" ").strip()
            
            # last 2 chars assumed to be state abbreviation
            state = location[-2:]
            
            company_url = (cols[3].find("a").get("href")  if cols[3].find("a") else "err")
            
            if any(k in role for k in keywords) and state in keyStates:
                all_rows.append({
                    "Company": cols[0].get_text(separator=" ").strip(),
                    "Location":cols[2].get_text(separator=" ").strip(),
                    "Role": role,
                    "URL": company_url
                })

    # 4. DATA ANALYSIS
    df = pd.DataFrame(all_rows)
    if not df.empty:
        validPositions = len(df)/len(rows) * 100
        print(f"✅ Successfully scanned the Data Science section.")
        print(f"📊 Total Roles: {len(rows)}")
        print(f"valid positions: {validPositions:.1f}%")
        print(f"Total valid positions: {len(df)}")
        
        # Displaying the Advanced Roles
        display(df)
        df.to_csv("internships.csv", index=False)
    else:
        print("Found the header, but no tables were found in that section.")
else:
    print("Error: Could not find the Data Science header in the full document.")

✅ Successfully scanned the Data Science section.
📊 Total Roles: 266
valid positions: 13.9%
Total valid positions: 37


,Company,Location,Role,URL
0,Mass General Brigham,"Boston, MA",behavioral neuroscience and computer vision co...,https://massgeneralbrigham.wd1.myworkdayjobs.c...
1,Nationwide Children's Hospital,"Columbus, OH",it r&1 data science intern,https://nationwidechildrens.wd5.myworkdayjobs....
2,Draper,"Cambridge, MA Reston, VA",market intelligence intern,https://draper.wd5.myworkdayjobs.com/Draper_Ca...
3,Syneos Health,"Morrisville, NC",data science / analytics intern - technology s...,https://syneoshealth.wd12.myworkdayjobs.com/Sy...
4,Johnson Controls,"Milwaukee, WI",marketing data & analytics intern,https://jci.wd5.myworkdayjobs.com/JCI/job/Milw...
5,🔥 ByteDance,"San Jose, CA",research scientist intern - seed responsible ai,https://jobs.bytedance.com/en/position/7642762...
6,Eurofins,"Lancaster, PA",business intelligence intern,https://jobs.smartrecruiters.com/Eurofins/7440...
7,Primetals Technologies,"Lake Mary, FL",data science intern - data platforms,https://mhicareers.com/job/Lake-Mary-Data-Scie...
8,Lila Sciences,"Cambridge, MA",ai resident - material science,https://job-boards.greenhouse.io/lilasciences/...
9,Smartly.io,"Chicago, IL",marketing science intern,https://job-boards.greenhouse.io/smartlyio/job...


In [29]:
import matplotlib.pyplot as plt
